# 01 — Exploratory Data Analysis

**Dataset**: Yakimov et al. (2024) — Fingernail bed images with hemoglobin (Hb) labels.

## Objectives
1. Load and inspect the dataset structure
2. Analyze Hb value distribution (g/dL)
3. Assess image quality and resolution consistency
4. Visualize sample fingernail images across Hb ranges

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

# Configuration
DATA_ROOT = Path("../data/raw")  # Adjust to your dataset location
METADATA_CSV = DATA_ROOT / "metadata.csv"

print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Exists: {DATA_ROOT.exists()}")

## 1. Load Metadata

In [ ]:
# Load the metadata CSV
# Expected columns: image_path, hb_value (g/dL), patient_id, finger, lighting, ...
if METADATA_CSV.exists():
    df = pd.read_csv(METADATA_CSV)
    print(f"Loaded {len(df)} samples")
    display(df.head(10))
    display(df.describe())
else:
    print(f"Metadata not found at {METADATA_CSV}")
    print("Please download the dataset and place metadata.csv in data/raw/")
    df = pd.DataFrame()  # empty placeholder

## 2. Hemoglobin Distribution

In [ ]:
if not df.empty and "hb_value" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    axes[0].hist(df["hb_value"], bins=30, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Hemoglobin (g/dL)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Hb Value Distribution")
    axes[0].axvline(x=12.0, color="red", linestyle="--", label="Anemia threshold (female)")
    axes[0].axvline(x=13.0, color="orange", linestyle="--", label="Anemia threshold (male)")
    axes[0].legend()

    # Box plot
    sns.boxplot(y=df["hb_value"], ax=axes[1])
    axes[1].set_ylabel("Hemoglobin (g/dL)")
    axes[1].set_title("Hb Value Box Plot")

    plt.tight_layout()
    plt.show()

    # Key stats
    print(f"Mean Hb:   {df['hb_value'].mean():.2f} g/dL")
    print(f"Median Hb: {df['hb_value'].median():.2f} g/dL")
    print(f"Std Hb:    {df['hb_value'].std():.2f} g/dL")
    print(f"Range:     [{df['hb_value'].min():.1f}, {df['hb_value'].max():.1f}] g/dL")
else:
    print("Skipping — no data loaded.")

## 3. Image Quality Assessment

In [ ]:
def get_image_stats(image_path: Path) -> dict:
    """Compute basic image statistics."""
    img = Image.open(image_path)
    arr = np.array(img)
    return {
        "width": img.width,
        "height": img.height,
        "channels": arr.shape[2] if arr.ndim == 3 else 1,
        "mean_brightness": arr.mean(),
        "std_brightness": arr.std(),
    }


if not df.empty and "image_path" in df.columns:
    # Sample a subset for quick profiling
    sample = df.sample(min(100, len(df)), random_state=42)
    stats = []
    for _, row in sample.iterrows():
        img_path = DATA_ROOT / row["image_path"]
        if img_path.exists():
            stats.append(get_image_stats(img_path))

    if stats:
        stats_df = pd.DataFrame(stats)
        display(stats_df.describe())
    else:
        print("No images found at expected paths.")
else:
    print("Skipping — no data loaded.")

## 4. Sample Visualization

In [ ]:
def show_samples(dataframe: pd.DataFrame, n: int = 8):
    """Display a grid of sample fingernail images with Hb labels."""
    sample = dataframe.sample(min(n, len(dataframe)), random_state=42)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = axes.flatten()

    for i, (_, row) in enumerate(sample.iterrows()):
        img_path = DATA_ROOT / row["image_path"]
        if img_path.exists():
            img = Image.open(img_path)
            axes[i].imshow(img)
            axes[i].set_title(f"Hb = {row['hb_value']:.1f} g/dL", fontsize=11)
        else:
            axes[i].text(0.5, 0.5, "Not found", ha="center", va="center")
        axes[i].axis("off")

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Sample Fingernail Images", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


if not df.empty and "image_path" in df.columns:
    show_samples(df)
else:
    print("Skipping — no data loaded.")

## Next Steps

- Proceed to `02_feature_extraction.ipynb` for ROI cropping and color-space analysis.
- Check for class imbalance and consider stratified splits in `03_baseline_models.ipynb`.